In [1]:
```python
# ============================================================
# TUSCAN SUBURBIA ITEMS
# NLP + TF-IDF + K-MEANS CLUSTERING
# ============================================================

# ============================================================
# 1. INSTALL REQUIRED PACKAGES
# ============================================================

# Run this cell once if the packages are not installed.
# Uncomment the following line:

# !pip install pandas numpy matplotlib seaborn scikit-learn nltk


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

warnings.filterwarnings("ignore")

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD


# ============================================================
# 3. DOWNLOAD NLTK DATA
# ============================================================

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


# ============================================================
# 4. LOAD DATASET
# ============================================================

FILE_NAME = "tuscan-suburbia-items.csv"

df = pd.read_csv(FILE_NAME)

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 5. BASIC DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


# ============================================================
# 6. REMOVE DUPLICATES
# ============================================================

df = df.drop_duplicates().reset_index(drop=True)

print("Rows after removing duplicates:", len(df))


# ============================================================
# 7. IDENTIFY TEXT COLUMN
# ============================================================

possible_text_columns = [
    "description",
    "Description",
    "text",
    "Text",
    "title",
    "Title",
    "name",
    "Name",
    "item",
    "Item",
    "content",
    "Content",
    "product",
    "Product",
    "category",
    "Category"
]

text_column = None

# First, look for a likely text column
for column in possible_text_columns:
    if column in df.columns:
        text_column = column
        break


# If no obvious text column exists,
# automatically select the object/string column
if text_column is None:

    object_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()

    if len(object_columns) == 0:
        raise ValueError(
            "No text/string column was found in the dataset."
        )

    # Choose the column with the greatest average text length
    text_column = max(
        object_columns,
        key=lambda column:
        df[column].fillna("").astype(str).str.len().mean()
    )


print("\n" + "=" * 70)
print("TEXT COLUMN")
print("=" * 70)

print("Selected text column:", text_column)

print("\nSample text:")
display(df[[text_column]].head(10))


# ============================================================
# 8. CREATE TEXT DATA
# ============================================================

df[text_column] = df[text_column].fillna("").astype(str)


# ============================================================
# 9. NLP TEXT CLEANING
# ============================================================

stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()


def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Keep only letters
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords
    words = [
        word
        for word in words
        if word not in stop_words
    ]

    # Remove very short words
    words = [
        word
        for word in words
        if len(word) > 2
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)


# Apply NLP cleaning
df["clean_text"] = df[text_column].apply(clean_text)


print("\n" + "=" * 70)
print("CLEANED TEXT")
print("=" * 70)

display(
    df[[text_column, "clean_text"]].head(10)
)


# ============================================================
# 10. REMOVE EMPTY TEXT RECORDS
# ============================================================

before = len(df)

df = df[
    df["clean_text"].str.strip() != ""
].copy()

df = df.reset_index(drop=True)

after = len(df)

print("\nRemoved empty text records:", before - after)
print("Remaining records:", after)


# ============================================================
# 11. TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(
    df["clean_text"]
)

print("\n" + "=" * 70)
print("TF-IDF")
print("=" * 70)

print("Number of documents:", X.shape[0])
print("Number of TF-IDF features:", X.shape[1])


# ============================================================
# 12. GET FEATURE NAMES
# ============================================================

terms = vectorizer.get_feature_names_out()

print("\nFirst 30 TF-IDF terms:")
print(terms[:30])


# ============================================================
# 13. FIND OPTIMAL NUMBER OF CLUSTERS
# ============================================================

print("\n" + "=" * 70)
print("FINDING OPTIMAL NUMBER OF CLUSTERS")
print("=" * 70)

# Maximum K depends on the number of records
max_k = min(10, len(df) - 1)

if max_k < 2:
    raise ValueError(
        "The dataset needs at least 3 records for clustering."
    )

k_values = range(2, max_k + 1)

inertia_values = []
silhouette_values = []


for k in k_values:

    print(f"Testing K = {k}...")

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    # Inertia
    inertia_values.append(
        model.inertia_
    )

    # Silhouette score
    silhouette = silhouette_score(
        X,
        labels
    )

    silhouette_values.append(
        silhouette
    )


# ============================================================
# 14. ELBOW METHOD
# ============================================================

plt.figure(figsize=(9, 5))

plt.plot(
    list(k_values),
    inertia_values,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")

plt.xticks(list(k_values))
plt.grid(True)

plt.show()


# ============================================================
# 15. SILHOUETTE SCORE
# ============================================================

plt.figure(figsize=(9, 5))

plt.plot(
    list(k_values),
    silhouette_values,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")

plt.xticks(list(k_values))
plt.grid(True)

plt.show()


# ============================================================
# 16. SELECT BEST K
# ============================================================

best_k = list(k_values)[
    np.argmax(silhouette_values)
]

best_silhouette = max(
    silhouette_values
)

print("\n" + "=" * 70)
print("BEST NUMBER OF CLUSTERS")
print("=" * 70)

print("Best K:", best_k)
print(
    "Best silhouette score:",
    round(best_silhouette, 4)
)


# ============================================================
# 17. TRAIN FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20,
    max_iter=500
)

df["cluster"] = kmeans.fit_predict(X)


print("\n" + "=" * 70)
print("K-MEANS COMPLETED")
print("=" * 70)

print("Number of clusters:", best_k)


# ============================================================
# 18. COUNT ITEMS IN EACH CLUSTER
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("\nItems per cluster:")
display(
    cluster_counts.to_frame(
        name="Number of Items"
    )
)


# ============================================================
# 19. VISUALISE CLUSTER SIZES
# ============================================================

plt.figure(figsize=(9, 5))

plt.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

plt.xlabel("Cluster")
plt.ylabel("Number of Items")
plt.title("Number of Items in Each Cluster")

plt.show()


# ============================================================
# 20. FIND TOP WORDS FOR EACH CLUSTER
# ============================================================

print("\n" + "=" * 70)
print("TOP WORDS IN EACH CLUSTER")
print("=" * 70)


order_centroids = (
    kmeans.cluster_centers_
    .argsort()[:, ::-1]
)


cluster_keywords = {}


for cluster_number in range(best_k):

    top_indices = (
        order_centroids[
            cluster_number,
            :20
        ]
    )

    top_words = [
        terms[index]
        for index in top_indices
    ]

    cluster_keywords[
        cluster_number
    ] = top_words

    print(
        f"\nCluster {cluster_number}"
    )

    print(
        ", ".join(top_words)
    )


# ============================================================
# 21. CREATE CLUSTER SUMMARY
# ============================================================

summary = []

for cluster_number in range(best_k):

    number_of_items = (
        df["cluster"] == cluster_number
    ).sum()

    top_words = cluster_keywords[
        cluster_number
    ]

    summary.append({
        "Cluster": cluster_number,
        "Number of Items": number_of_items,
        "Top Keywords": ", ".join(
            top_words[:10]
        )
    })


cluster_summary = pd.DataFrame(
    summary
)

print("\n" + "=" * 70)
print("CLUSTER SUMMARY")
print("=" * 70)

display(cluster_summary)


# ============================================================
# 22. SHOW EXAMPLES FROM EACH CLUSTER
# ============================================================

print("\n" + "=" * 70)
print("EXAMPLES FROM EACH CLUSTER")
print("=" * 70)


for cluster_number in range(best_k):

    print("\n")
    print("=" * 70)
    print(
        f"CLUSTER {cluster_number}"
    )
    print("=" * 70)

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    # Display up to 10 examples
    display(
        cluster_data[
            [text_column, "clean_text"]
        ].head(10)
    )


# ============================================================
# 23. REDUCE TF-IDF DIMENSIONS FOR VISUALISATION
# ============================================================

# TruncatedSVD works efficiently with sparse TF-IDF matrices.

svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2d = svd.fit_transform(X)


# ============================================================
# 24. CREATE VISUALISATION DATAFRAME
# ============================================================

visualisation_df = pd.DataFrame({
    "Component 1": X_2d[:, 0],
    "Component 2": X_2d[:, 1],
    "Cluster": df["cluster"].values
})


# ============================================================
# 25. PLOT K-MEANS CLUSTERS
# ============================================================

plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=visualisation_df,
    x="Component 1",
    y="Component 2",
    hue="Cluster",
    palette="tab10",
    s=80
)

plt.title(
    "K-Means Clusters Based on NLP Features"
)

plt.xlabel("Component 1")
plt.ylabel("Component 2")

plt.legend(
    title="Cluster",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()

plt.show()


# ============================================================
# 26. ADD REDUCED DIMENSIONS TO DATASET
# ============================================================

df["component_1"] = X_2d[:, 0]
df["component_2"] = X_2d[:, 1]


# ============================================================
# 27. SHOW FINAL DATASET
# ============================================================

print("\n" + "=" * 70)
print("FINAL CLUSTERED DATASET")
print("=" * 70)

display(df.head(20))


# ============================================================
# 28. SORT DATA BY CLUSTER
# ============================================================

df_sorted = df.sort_values(
    by="cluster"
).reset_index(drop=True)

display(
    df_sorted.head(20)
)


# ============================================================
# 29. SAVE CLUSTERED DATASET
# ============================================================

OUTPUT_FILE = (
    "tuscan-suburbia-items-clustered.csv"
)

df_sorted.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nClustered dataset saved to: {OUTPUT_FILE}"
)


# ============================================================
# 30. SAVE CLUSTER SUMMARY
# ============================================================

SUMMARY_FILE = (
    "tuscan-suburbia-cluster-summary.csv"
)

cluster_summary.to_csv(
    SUMMARY_FILE,
    index=False
)

print(
    f"Cluster summary saved to: {SUMMARY_FILE}"
)


# ============================================================
# 31. SAVE CLUSTER KEYWORDS
# ============================================================

keyword_rows = []

for cluster_number in range(best_k):

    for rank, word in enumerate(
        cluster_keywords[cluster_number],
        start=1
    ):

        keyword_rows.append({
            "Cluster": cluster_number,
            "Rank": rank,
            "Keyword": word
        })


keyword_df = pd.DataFrame(
    keyword_rows
)

KEYWORD_FILE = (
    "tuscan-suburbia-cluster-keywords.csv"
)

keyword_df.to_csv(
    KEYWORD_FILE,
    index=False
)

print(
    f"Cluster keywords saved to: {KEYWORD_FILE}"
)


# ============================================================
# 32. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    "Original dataset:",
    FILE_NAME
)

print(
    "Text column:",
    text_column
)

print(
    "Number of records:",
    len(df)
)

print(
    "Number of TF-IDF features:",
    X.shape[1]
)

print(
    "Optimal number of clusters:",
    best_k
)

print(
    "Silhouette score:",
    round(best_silhouette, 4)
)

print("\nCluster sizes:")

for cluster_number in range(best_k):

    count = (
        df["cluster"] == cluster_number
    ).sum()

    print(
        f"Cluster {cluster_number}: {count} items"
    )

print("\nOutput files:")
print(
    "1.",
    OUTPUT_FILE
)

print(
    "2.",
    SUMMARY_FILE
)

print(
    "3.",
    KEYWORD_FILE
)

print("\nAnalysis complete!")
```


SyntaxError: invalid syntax (1231922769.py, line 1)